# LangGraph G14 — Scheduled runs
Every run so far started with a user message. Real assistants also run **on a schedule**: every
morning, find students whose attendance has dropped below 75% and warn them. Nobody is typing,
nobody is watching, and the run may be triggered twice by a flaky scheduler. Three rules follow:

```text
1. A tick is a synthetic input, not a chat turn: {"tick_id": "2026-09-05T07:00"} with its own thread
2. Unattended runs may READ freely; WRITES are queued for a human (interrupt) and reviewed later
3. Ticks are idempotent: a ledger of processed tick ids means a duplicate trigger does nothing
```

```text
cron tick -> find_at_risk (deterministic) -> draft_emails (model) -> approval (interrupt, queued) -> send -> END
```

In production the tick comes from system cron, a scheduler library, or LangGraph Platform's cron
jobs calling the deployed graph. Here a small loop plays the scheduler so the mechanics are visible.

### Step 1 — The watcher graph: read, draft, queue the write

The graph pauses at `approval` like the records desk in G8, but now nobody resumes it
immediately: the interrupt simply stays in the checkpoint until a staff member reviews it.

In [ ]:
class WatchState(TypedDict, total=False):                   # ours
    tick_id: str
    at_risk: list
    drafts: list
    sent: list

def find_at_risk(state: WatchState):                        # ours: deterministic, no model
    return {"at_risk": [sid for sid, s in STUDENTS.items() if s["attendance"] < 75]}

def draft_emails(state: WatchState):                        # ours: one model call per student
    drafts = []
    for sid in state["at_risk"]:
        student = STUDENTS[sid]
        reply = model.invoke([SystemMessage("Draft a short email to a student whose attendance is below 75%. Be kind and clear."), HumanMessage(f"Student: {student['name']} ({sid}), attendance {student['attendance']}%.")])   # LangChain
        drafts.append({"to": sid, "subject": "Attendance warning", "body": text_of(reply)})
    return {"drafts": drafts}

def approval_queue(state: WatchState):                      # ours: queue the writes for a human
    approved = interrupt({"tick_id": state["tick_id"], "question": "Send these emails?", "drafts": state["drafts"]})   # LangGraph: waits in the checkpoint
    return {} if approved else {"drafts": []}

def send_drafts(state: WatchState):                         # ours: the write, only after approval
    sent = [json.loads(send_email.invoke({"to": d["to"], "subject": d["subject"], "body": d["body"]}))["to"] for d in state["drafts"]]   # LangChain: call the tool directly
    return {"sent": sent}

g = StateGraph(WatchState)
for name, fn in [("find_at_risk", find_at_risk), ("draft_emails", draft_emails), ("approval", approval_queue), ("send", send_drafts)]:
    g.add_node(name, fn)
g.add_edge(START, "find_at_risk"); g.add_edge("find_at_risk", "draft_emails"); g.add_edge("draft_emails", "approval"); g.add_edge("approval", "send"); g.add_edge("send", END)
attendance_watch = g.compile(checkpointer=InMemorySaver())  # LangGraph: the queue lives in the checkpoints
print(attendance_watch.get_graph().draw_mermaid())

### Step 2 — A scheduler that ticks, with an idempotency ledger

Each tick gets its own thread named after the tick id. The ledger makes a repeated tick a no-op.
The scheduler never waits for a human: it leaves paused threads behind.

In [ ]:
PROCESSED_TICKS = {}                                        # ours: tick_id -> thread config (the idempotency ledger)

def on_tick(tick_id):                                       # ours: what the scheduler calls
    if tick_id in PROCESSED_TICKS:
        print(f"  tick {tick_id}: already processed, skipping (duplicate trigger)")
        return
    config = {"configurable": {"thread_id": f"attendance-watch-{tick_id}"}}   # LangGraph: one thread per tick
    result = attendance_watch.invoke({"tick_id": tick_id}, config)
    PROCESSED_TICKS[tick_id] = config
    status = "queued for approval" if "__interrupt__" in result else "finished"
    print(f"  tick {tick_id}: {len(result['at_risk'])} at-risk student(s), {len(result['drafts'])} draft(s) -> {status}")

print("scheduler ticks (simulated: a loop instead of cron):")
for tick_id in ["2026-09-05T07:00", "2026-09-05T07:00", "2026-09-06T07:00"]:   # the second is a duplicate delivery
    on_tick(tick_id)
print("emails actually sent so far:", len(EMAIL_OUTBOX))

### Step 3 — Later, a staff member reviews the queue

The pending approvals are found by asking each thread what it is waiting for. Approving one
resumes it at the `approval` node and the emails go out; the other stays queued.

In [ ]:
pending = [(tick, cfg) for tick, cfg in PROCESSED_TICKS.items() if attendance_watch.get_state(cfg).next == ("approval",)]   # LangGraph: what is each thread waiting for?
print("threads waiting for approval:", [tick for tick, _ in pending])
for tick, cfg in pending:
    drafts = attendance_watch.get_state(cfg).tasks[0].interrupts[0].value["drafts"]   # LangGraph: the interrupt payload
    print(f"  {tick}: {[d['to'] for d in drafts]} -> {drafts[0]['body'][:70]}...")

tick, cfg = pending[0]
done = attendance_watch.invoke(Command(resume=True), cfg)     # LangGraph: the human approves the first tick
print(f"\napproved {tick}: sent to {done['sent']}")
print("emails in the outbox:", len(EMAIL_OUTBOX), "| still queued:", [t for t, c in PROCESSED_TICKS.items() if attendance_watch.get_state(c).next == ('approval',)])

### Recap

- **Problem seen:** every run needed a user, and an unattended run could act twice or act without review.
- **Layer added:** a watcher graph triggered by synthetic ticks on their own threads, queued approvals that wait in checkpoints, and an idempotency ledger.
- **Evidence:** two ticks ran and one duplicate was skipped; nothing was sent until a person approved; the second tick is still waiting.